# 05 - Spatial statistics (Phase 5)

Global and local **Moran's I**, **Getis-Ord Gi\***, **Emerging Hot Spot
Analysis**, the OLS -> spatial lag/error -> **GWR/MGWR** ladder, a **MAUP**
comparison across two aggregation units, and **landscape metrics** on the
green-space class.

## What this notebook is for

Phase 4 measured *how fast* Colombo is warming and found that only MODIS Terra
night could resolve it. This notebook asks the other questions: **where** the
heat concentrates, whether those concentrations are **moving**, and **what
explains them** once spatial dependence is modelled instead of ignored.

It also discharges a debt. Phase 3's driver OLS was fitted to 5 000 spatially
autocorrelated sampled pixels, so its standard errors are anti-conservative by
construction - `config/params.yaml` calls it "a screening device, not
inference". Nothing here inherits that flaw, and nothing here "confirms" those
coefficients.

## This notebook runs in TWO PARTS

| | What it does | Needs Earth Engine? |
|---|---|---|
| **Part 1** | exports zone geometry, per-zone covariates, annual series and green rasters | yes |
| *(wait)* | you download the results from Drive into `data/interim/` | - |
| **Part 2** | every statistic and figure, computed locally in numpy/PySAL | **no** |

The split is the Phase 4 pattern and it is not stylistic: Colab runs 10-13
established that no interactive question about a district-wide composite graph
is affordable. Part 1 asks Earth Engine only for things a batch `Export` can
carry.

## Three decisions taken before writing this, recorded here

| # | Decision | Why |
|---|---|---|
| 1 | Epoch cluster maps use the **pooled** `landsat_dry` series | Moran's I, LISA and Gi\* are WITHIN-epoch statistics computed on deviations from that epoch's own mean, so the sensor steps Phase 4 measured (L5-L7 +1.78, L7-L8 -2.48 degC) cancel exactly as they do in SUHII. **No epoch-to-epoch magnitude may be quoted from these maps** - only cluster geography. Step 9 re-runs the 2020s on the single-sensor series to check that empirically. |
| 2 | EHSA runs on **both** `landsat_oli_dry` (12 bins, 100 m) and `terra_night` (26 bins, 1 km) | Neither is sufficient alone: one has the spatial detail, the other the temporal power. Each zone ships its own detection limit. |
| 3 | GN gets the full ladder; **DS gets only what n=13 supports** | 13 units against 6 predictors leaves a GWR with effectively no degrees of freedom. It still returns numbers and still makes a colourful map. Refusing to fit it, and saying why, is the result. |

## Non-negotiable caveats

1. This is **land surface temperature**, never air temperature.
2. Every zonal mean ships its pixel count.
3. These statistics describe **polygons**, not pixels and not people. Reading a
   zonal coefficient as an individual-level relationship is the ecological
   fallacy.
4. 557 simultaneous local tests are 557 tests. Every local statistic here
   carries a Benjamini-Hochberg adjusted p, and both counts are reported.

---
# PART 1 - export

Everything here is a batch task or a cheap lookup. Nothing interrogates a
composite graph interactively, and any step that turns out to be too expensive
prints a note and continues rather than aborting the run.

In [ ]:
# COLAB: RUN THIS CELL
# Clone the repo on first run; fast-forward pull on later runs.
import os

REPO_URL = "https://github.com/Dineth0627/colombo_uhi.git"
REPO_DIR = "/content/" + REPO_URL.rstrip("/").removesuffix(".git").rsplit("/", 1)[-1]

if not os.path.isdir(os.path.join(REPO_DIR, ".git")):
    !git clone {REPO_URL} {REPO_DIR}

%cd {REPO_DIR}
!git pull --ff-only

# Which revision is actually on disk. Quote this if a result looks impossible.
!git --no-pager log -1 --format="HEAD %h %s (%ci)"

In [ ]:
# COLAB: RUN THIS CELL  (skip if you already ran notebook 00-04 in this runtime)
%pip install -q -r requirements.txt
print("\nIf Colab asked to RESTART the runtime: Runtime > Restart session,")
print("then re-run this notebook FROM THE CLONE CELL (skip this pip cell).")

In [ ]:
# COLAB: RUN THIS CELL
# Load params (single source of truth) and initialise Earth Engine.
import sys

sys.path.insert(0, os.path.abspath("src"))

# Drop any already-imported colombo_uhi modules BEFORE importing. Without this,
# re-running the notebook in a live runtime keeps the version cached in
# sys.modules from the previous run: `git pull` updates the files on disk but the
# import silently returns the OLD code, so new functions appear to not exist
# (AttributeError) and fixed bugs appear unfixed. This cost a full run in Phase 1d.
for _name in [m for m in list(sys.modules) if m == "colombo_uhi" or m.startswith("colombo_uhi.")]:
    del sys.modules[_name]

from colombo_uhi import load_params
from colombo_uhi.auth import init_ee

params = load_params()
project = init_ee()
print("Earth Engine initialised with project:", project)
print()
for _key in ("lst_not_air_temp", "within_epoch_only", "zonal_not_pixel", "fdr_dependence"):
    print("CAVEAT:", " ".join(params["caveats"][_key].split()))
    print()

## Step 0 - prove the loaded code is current

The guard below names functions introduced by the **most recent** revision of
each module. Listing only functions that also existed in the previous revision
makes it vacuous: it passes, and you get old figures from new notebook cells
with no error anywhere. That happened in Phase 3, run 8.

In [ ]:
# COLAB: RUN THIS CELL
# Every import Parts 1 AND 2 need, in one place. Part 2 is meant to be run as a
# separate session, so it must not depend on an import that happens to live in a
# Part 1 cell.
import glob
import time
import warnings

import ee
import numpy as np
import pandas as pd
from IPython.display import Image, display

from colombo_uhi import aoi, composites, exports, landcover, spatial_stats, trends, uhi_metrics, viz

_required = {
    "spatial_stats": [
        # Everything below arrived with Phase 5; the module was a docstring stub
        # before it, so any AttributeError here means a stale checkout.
        "build_weights", "weights_report", "contiguity_neighbours",
        "global_morans_i", "local_morans", "gi_star", "gi_star_confidence_class",
        "space_time_bins", "gi_star_panel", "classify_zone_pattern",
        "classify_emerging_hotspots", "ehsa_power_check",
        "unit_noise_detection_limit", "require_estimable", "ols_fit",
        "lagrange_multiplier_tests", "lm_decision", "build_model_frame",
        "variance_inflation_factors", "spatial_lag_model", "spatial_error_model",
        "gwr_model", "mgwr_model", "maup_comparison",
        "patch_labels", "aggregation_index", "landscape_metrics",
        "landscape_metrics_by_zone", "build_landscape_frame",
        "division_geometry_collection", "export_division_geojson",
        "read_zone_geodataframe", "distance_to_coast", "population_density",
        "covariate_stack", "zone_covariate_collection", "export_zone_covariates",
        "read_zone_covariates", "green_class_image",
        "esda_cross_check", "spreg_cross_check",
    ],
    "viz": [
        "spatial_palette", "build_cluster_map_figure", "build_hotspot_map_figure",
        "build_ehsa_map_figure", "build_gwr_coefficient_figure",
        "build_maup_table_figure", "build_landscape_change_figure",
    ],
    "trends": ["zonal_annual_series", "minimum_detectable_slope", "benjamini_hochberg"],
    "uhi_metrics": ["zonal_by_division", "epoch_composite", "driver_stack"],
    "exports": ["export_name", "table_to_drive", "image_to_drive", "describe_tasks"],
}
_absent = {
    name: [f for f in funcs if not hasattr(globals()[name], f)]
    for name, funcs in _required.items()
}
_absent = {k: v for k, v in _absent.items() if v}
if _absent:
    raise RuntimeError(
        f"colombo_uhi modules are STALE, missing {_absent}.\n"
        f"  spatial_stats loaded from: {spatial_stats.__file__}\n"
        "Fix, in order:\n"
        "  1. MOST LIKELY: local changes are COMMITTED BUT NOT PUSHED. This\n"
        "     notebook runs against the pushed repo. Check the HEAD line from\n"
        "     the clone cell, then git push and re-run FROM THE CLONE CELL.\n"
        "  2. Runtime > Restart session, then re-run from the CLONE cell.\n"
        "  3. If you uploaded this .ipynb by hand rather than opening it from\n"
        "     the repo, the notebook and src/ can be at DIFFERENT revisions."
    )
if "levels" not in params["spatial_stats"]:
    raise RuntimeError(
        "config/params.yaml has no spatial_stats.levels - your checkout "
        "predates Phase 5. Re-run the CLONE cell."
    )
print("PASS: all Phase 5 functions and params are present.")


# Run an interactive Earth Engine call; degrade to a note on a memory error.
# Everything that matters goes out through batch Export tasks, so an interactive
# call failing costs a convenience, never a result.
def _try_ee(label, call, default=None):
    try:
        return call()
    except ee.EEException as error:
        if "memory" not in str(error).lower():
            raise
        print(f"SKIPPED ({label}): exceeded the interactive memory limit.")
        print("  Not a failure - the batch exports carry the real products.")
        return default


SS = params["spatial_stats"]
LEVELS = list(SS["levels"])
EPOCHS = list(params["uhi"]["utfvi"]["epochs"])
EPOCH_SOURCE = SS["epochs_source"]
SENSITIVITY_SOURCE = SS["epoch_sensitivity_source"]
EPOCH_SCALE = int(SS["epoch_scale_m"])
RESPONSE = SS["response_band"]
PREDICTORS = list(SS["regression"]["predictors"])
EHSA_SOURCES = list(SS["ehsa"]["sources"])

# The green-class products, built ONCE here so Part 1's export loop and Part 2's
# read-back loop cannot drift apart and look for a file the other never wrote.
GREEN_SCHEMES = [
    ("dynamic_world", int(_year)) for _year in SS["landscape"]["dynamic_world_years"]
]
_WC_ASSET = SS["landscape"]["worldcover_asset"]
GREEN_SCHEMES.append(
    ("worldcover", int(params["datasets"][_WC_ASSET]["nominal_year"]))
)

# The sensitivity run shares its level and epoch with the pooled run, so it MUST
# carry its own export suffix or the second task renders to the same Drive
# filename and silently overwrites the first.
SENSITIVITY_SUFFIX = f"gn_{EPOCHS[-1]}_sensitivity"

print()
print("Levels     :", LEVELS)
print("Epochs     :", EPOCHS, "from", EPOCH_SOURCE, f"at {EPOCH_SCALE} m")
print("Predictors :", PREDICTORS)
print("EHSA       :", EHSA_SOURCES)
print("Green      :", GREEN_SCHEMES)

## Step 1 - PROBE the PySAL stack

Phase 4 settled the Earth Engine reducer band names empirically rather than
trusting documentation, and that is why run 11 cost one cell instead of one run.
The same applies here.

`spatial_stats` computes Moran's I, LISA and Gi\* **analytically in numpy**
rather than calling `esda`. That is deliberate - it keeps the numerical core
unit-testable in an environment without PySAL, makes the permutation p-values
reproducible from a seed, and removes a dependency on API details that vary
between releases. **But a re-implementation is only safe if it is checked.**

This step prints the installed versions and the real call signatures, then
cross-validates every statistic against `esda` and `spreg` on a lattice where
the answers are known. **Read the `abs_diff` column: everything must be
essentially zero.** If it is not, stop and report both tables before trusting
any map below.

One row needs its `note` read alongside it. **Local Moran's I has two published
normalisations** - Anselin (1995) divides by the population second moment,
GeoDa and `esda` by `(n-1)` - so the two differ by a single constant factor
identical for every zone. That constant cannot change a quadrant, a permutation
p-value or a cluster map, so the comparison is made after rescaling, the ratio
is printed, and a separate row asserts the **quadrant labels agree exactly**.
Global Moran's I and Gi\* have one convention each and are compared directly.

In [ ]:
# COLAB: RUN THIS CELL
import inspect

import esda
import libpysal
import mgwr
import spreg

for _module in (libpysal, esda, spreg, mgwr):
    print(f"{_module.__name__:10s}", getattr(_module, "__version__", "(no __version__)"))
print()
for _label, _target in (
    ("esda.Moran_Local", esda.Moran_Local.__init__),
    ("esda.G_Local", esda.G_Local.__init__),
    ("mgwr.sel_bw.Sel_BW", __import__("mgwr.sel_bw", fromlist=["Sel_BW"]).Sel_BW.__init__),
    ("mgwr.gwr.GWR", __import__("mgwr.gwr", fromlist=["GWR"]).GWR.__init__),
):
    _sig = inspect.signature(_target)
    print(f"{_label}{str(_sig).replace('self, ', '')}")
    print()

# A 6x6 rook lattice with a north-south gradient: Moran's I, LISA quadrants and
# Gi* all have known signs on it.
_side = 6
_nb = [[] for _ in range(_side * _side)]
for _r in range(_side):
    for _c in range(_side):
        for _dr, _dc in ((1, 0), (-1, 0), (0, 1), (0, -1)):
            _rr, _cc = _r + _dr, _c + _dc
            if 0 <= _rr < _side and 0 <= _cc < _side:
                _nb[_r * _side + _c].append(_rr * _side + _cc)
_W = spatial_stats.row_standardise(spatial_stats.neighbours_to_matrix(_nb))
_values = np.array([30.0 + 2.0 * _r + 0.1 * _c for _r in range(_side) for _c in range(_side)])

print("=== our analytic statistics vs esda ===")
_esda_check = spatial_stats.esda_cross_check(_values, _W, params)
display(_esda_check)

print("=== our OLS + Lagrange Multiplier tests vs spreg ===")
_rng = np.random.default_rng(0)
_X = _rng.normal(size=(_side * _side, 2))
_y = 1.0 + 2.0 * _X[:, 0] - _X[:, 1] + _rng.normal(scale=0.4, size=_side * _side)
_spreg_check = spatial_stats.spreg_cross_check(_y, _X, ["a", "b"], _W)
display(_spreg_check)

_worst = max(_esda_check["abs_diff"].max(), _spreg_check["abs_diff"].max())
print()
if _worst < 1e-6:
    print(f"PASS: the largest disagreement is {_worst:.2e}. The analytic")
    print("implementations agree with the reference libraries.")
else:
    print(f"*** STOP. Largest disagreement is {_worst:.3g}, which is NOT numerical")
    print("noise. Do not trust any map below until this is explained. Report the")
    print("two tables above verbatim.")

# LISA quadrant coding is a hard-coded convention in spatial_stats; confirm the
# installed esda agrees, because a silent relabel would invert a cluster map.
_ours = spatial_stats.local_morans(_values, _W, params, permutations=199)["quadrant"].to_numpy()
_w_ps = libpysal.weights.W(
    {i: [int(j) for j in np.flatnonzero(_W[i] > 0)] for i in range(_side * _side)},
    {i: [float(_W[i, j]) for j in np.flatnonzero(_W[i] > 0)] for i in range(_side * _side)},
    silence_warnings=True,
)
_theirs = esda.Moran_Local(_values, _w_ps, permutations=0).q
print()
print("LISA quadrant coding matches esda:", bool((_ours == _theirs).all()),
      "| mapping:", spatial_stats.LISA_QUADRANTS)

## Step 2 - geometries, and the zone polygons nothing has ever exported

Phase 5 inherited one structural gap: **no zone geometry had ever left Earth
Engine.** There is no GeoJSON, no centroid column, nothing `libpysal` can build
a weights matrix from. Closing that is the first export.

The polygons are simplified to `spatial_stats.geometry.simplify_m` and stripped
to the properties Phase 5 needs. That is not tidiness: `data/outputs/` is
committed, and an unsimplified 557-polygon COD-AB export with every attribute
is far larger than a repository should carry.

In [ ]:
# COLAB: RUN THIS CELL
district_fc = aoi.colombo_district(params)
WORK_REGION = district_fc.geometry()
CMC = aoi.cmc_boundary(params)

_counts = {}
for _level in LEVELS:
    _fc = aoi.gn_divisions(params) if _level == "gn" else aoi.ds_divisions(params)
    _counts[_level] = _fc.size().getInfo()
print("Zone counts:", _counts)

_expected = {
    "gn": params["aoi"]["expected_counts"]["gn_divisions"],
    "ds": params["aoi"]["expected_counts"]["ds_divisions"],
}
for _level, _n in _counts.items():
    _flag = "OK" if _n == _expected[_level] else "MISMATCH"
    print(f"  {_level}: {_n} (expected {_expected[_level]}) {_flag}")
print()
print("REMINDER: zones are keyed on the PCODE, never the name. GN names are not")
print("unique within Colombo District - matching on name pulls in same-named")
print("divisions from Dehiwala, Moratuwa and Kolonnawa.")

In [ ]:
# COLAB: RUN THIS CELL
TASKS = []

for _level in LEVELS:
    _task = spatial_stats.export_division_geojson(params, _level)
    TASKS.append(_task)
    print("submitted:", _task.status().get("description"))

## Step 3 - per-zone covariates, one epoch at a time

One `reduceRegions` over the whole multi-band stack per epoch per level, run
**inside a batch task**. That is the only reason it is possible: a district-wide
reduction over an epoch composite plus a reflectance composite plus GHSL,
WorldPop, SRTM and a distance transform is not an interactive question.

Two things travel with these numbers and must not be relabelled later:

* **WorldPop ends in 2020.** The population column is the 2020 layer whatever
  epoch it sits in.
* **GHSL is published in 5-year epochs**, so the built fraction is snapped down
  to the epoch containing the requested year.

In [ ]:
# COLAB: RUN THIS CELL
for _level in LEVELS:
    for _epoch in EPOCHS:
        _task = spatial_stats.export_zone_covariates(
            params, _level, _epoch, WORK_REGION
        )
        TASKS.append(_task)
        print("submitted:", _task.status().get("description"))

# The 2020s epoch again on the SINGLE-SENSOR series. If the cluster geography
# from this matches the pooled-series map, decision 1 in the header is
# empirically justified rather than merely argued. The distinct suffix is
# load-bearing: without it this task and the pooled one render to the SAME
# Drive filename and the second overwrites the first with no warning.
_sensitivity = spatial_stats.export_zone_covariates(
    params, "gn", EPOCHS[-1], WORK_REGION, source=SENSITIVITY_SOURCE,
    suffix=SENSITIVITY_SUFFIX,
)
TASKS.append(_sensitivity)
print("submitted:", _sensitivity.status().get("description"), "(sensitivity)")
print()
print(f"{len(TASKS)} task(s) queued so far.")

## Step 4 - the annual series EHSA needs

`trends.zonal_annual_series` is the Phase 4 function that already produced
`data/outputs/lst_by_gn_annual_2000_2025.csv`, batched four years per request.
It succeeded in run 18, so it is used interactively here rather than exported -
but it still goes through `_try_ee`.

Both sources run, and both are reported. `landsat_oli_dry` has the 100 m
intra-urban detail and 12 bins; `terra_night` has 26 bins and a single sensor
but 1 km pixels against a median GN division of about 1.25 km2. Neither may be
quoted alone.

In [ ]:
# COLAB: RUN THIS CELL
os.makedirs("data/interim", exist_ok=True)
os.makedirs("data/outputs", exist_ok=True)

EHSA_SERIES = {}
for _source in EHSA_SOURCES:
    # MODIS at 1 km cannot resolve a GN division, so terra_night is run at BOTH
    # levels and landsat_oli_dry only at GN.
    _levels = LEVELS if _source.startswith("terra") else ["gn"]
    for _level in _levels:
        _t0 = time.time()
        _frame = _try_ee(
            f"{_source} annual series at {_level}",
            lambda s=_source, l=_level: trends.zonal_annual_series(
                s, params, level=l, region=WORK_REGION, progress=True
            ),
        )
        if _frame is None or _frame.empty:
            print(f"  {_source}/{_level}: unavailable this run")
            continue
        EHSA_SERIES[(_source, _level)] = _frame
        _path = f"data/interim/ehsa_series_{_source}_{_level}.csv"
        _frame.to_csv(_path, index=False)
        print(f"  {_source}/{_level}: {len(_frame)} rows, "
              f"{_frame['year'].nunique()} years, {_frame['zone_id'].nunique()} zones "
              f"in {time.time() - _t0:.0f} s -> {_path}")

## Step 5 - the green-space rasters

Two Dynamic World dates so the deliverable is fragmentation **change**, plus
WorldCover 2021 as a cross-source check on a different sensor and legend. If the
two disagree wildly, the metric is measuring the classifier rather than the
city, and that is worth knowing before it reaches a recommendation.

Cropland is deliberately **excluded** from the green class in both schemes:
agricultural land is not urban green space for a greening-priority purpose.

In [ ]:
# COLAB: RUN THIS CELL
for _scheme, _year in GREEN_SCHEMES:
    _image = spatial_stats.green_class_image(
        _scheme, params, year=_year if _scheme == "dynamic_world" else None,
        region=WORK_REGION,
    ).clip(WORK_REGION)
    _task = exports.image_to_drive(
        _image, product="green_class", aoi="district", params=params,
        # bounds(), not the 557-vertex polygon: the vertex list travels with
        # every export request (Phase 4 lesson).
        region=WORK_REGION.bounds(),
        band_order=["green"],
        scale_m=int(SS["landscape"]["raster_scale_m"]),
        suffix=f"{_scheme}_{_year}",
    )
    TASKS.append(_task)
    print("submitted:", _task.status().get("description"),
          "| green codes:", spatial_stats.resolve_green_classes(_scheme, params))

print()
print(f"{len(TASKS)} task(s) queued to Drive folder "
      f"'{params['exports']['drive_folder']}'.")
print("Re-run the NEXT cell until every state reads COMPLETED.")

In [ ]:
# COLAB: RUN THIS CELL  (re-runnable - poll until every state reads COMPLETED)
exports.describe_tasks(TASKS)

---
# WAIT HERE

**Part 1 is done once every task in the status cell reads `COMPLETED`.**

Then get the files into `data/interim/`. Either:

```python
from google.colab import drive
drive.mount('/content/drive')
!cp /content/drive/MyDrive/colombo_uhi_exports/*.geojson data/interim/
!cp /content/drive/MyDrive/colombo_uhi_exports/*.csv data/interim/
!cp /content/drive/MyDrive/colombo_uhi_exports/*.tif data/interim/
```

or download them from Drive in a browser and upload into `data/interim/`.

**To run Part 2 as a separate session**, re-run only these, then continue below:

1. the **clone** cell,
2. the **pip** cell (skip if the runtime is fresh from Part 1),
3. the **params + auth** cell,
4. **Step 0** - it carries every import and constant Part 2 needs.

Steps 1-5 do not need re-running: Part 2 reads from `data/interim/`, not from
Earth Engine. Step 1's probe output is worth keeping, though - it is the
evidence that the analytic statistics agree with PySAL.

---
# PART 2 - analyse

No Earth Engine below this line. Everything is numpy, PySAL and matplotlib on
the downloaded files.

In [ ]:
# COLAB: RUN THIS CELL
# Locate the downloaded products and fail with an actionable message if absent.
import geopandas as gpd
import rasterio

GEOJSON = {}
for _level in LEVELS:
    _name = exports.export_name(
        f"{_level}_divisions", "district", params, res_m=EPOCH_SCALE
    )
    _path = f"data/interim/{_name}.geojson"
    GEOJSON[_level] = _path if os.path.exists(_path) else None

COVARIATE_CSV = {}
for _level in LEVELS:
    for _epoch in EPOCHS:
        _name = exports.export_name(
            "zone_covariates", "district", params, res_m=EPOCH_SCALE,
            suffix=f"{_level}_{_epoch}",
        )
        _path = f"data/interim/{_name}.csv"
        COVARIATE_CSV[(_level, _epoch)] = _path if os.path.exists(_path) else None

GREEN_TIF = {}
for _scheme, _year in GREEN_SCHEMES:
    _name = exports.export_name(
        "green_class", "district", params,
        res_m=int(SS["landscape"]["raster_scale_m"]), suffix=f"{_scheme}_{_year}",
    )
    _path = f"data/interim/{_name}.tif"
    GREEN_TIF[(_scheme, _year)] = _path if os.path.exists(_path) else None

print("Found in data/interim/:")
for _f in sorted(glob.glob("data/interim/*")):
    print("  ", _f, f"({os.path.getsize(_f) / 1024:.0f} KB)")
print()
for _label, _mapping in (("geometry", GEOJSON), ("covariates", COVARIATE_CSV),
                         ("green raster", GREEN_TIF)):
    _missing = [k for k, v in _mapping.items() if v is None]
    print(f"  {_label:14s}: {len(_mapping) - len(_missing)}/{len(_mapping)} present"
          + (f"  MISSING {_missing}" if _missing else ""))

if GEOJSON["gn"] is None:
    raise FileNotFoundError(
        "the GN geometry is missing, and nothing in Part 2 can run without it.\n"
        "1. Check every Part 1 task reads COMPLETED.\n"
        f"2. Copy the files from Drive folder "
        f"'{params['exports']['drive_folder']}' into data/interim/ - see the\n"
        "   WAIT HERE cell above for the exact commands."
    )

## Step 6 - zone geometry, weights, and the island report

**Read the island count before anything else.** A polygon with no contiguity
neighbour does not raise anywhere in the PySAL stack: it produces a statistic
computed over an empty neighbourhood that lands on the map looking like every
other value. Colombo's coast is ragged and the COD-AB polygon encloses the port
outer harbour, so this is a live risk here rather than a theoretical one.

The default policy attaches each island to its single nearest neighbour, and the
count is always reported so the repair is visible rather than assumed.

The geometry is also **reprojected to EPSG:32644**. Earth Engine writes GeoJSON
in EPSG:4326; contiguity would survive that, but centroid distances, KNN and
every GWR bandwidth would be in degrees, which is not a distance.

In [ ]:
# COLAB: RUN THIS CELL
ZONES, WEIGHTS, ZONE_IDS, WEIGHT_REPORT = {}, {}, {}, {}

for _level in LEVELS:
    if GEOJSON[_level] is None:
        print(f"{_level}: geometry missing, skipping")
        continue
    _gdf = spatial_stats.read_zone_geodataframe(GEOJSON[_level], params, _level)
    _matrix, _ids, _report = spatial_stats.build_weights(_gdf, params)
    ZONES[_level] = _gdf
    WEIGHTS[_level] = _matrix
    ZONE_IDS[_level] = _ids
    WEIGHT_REPORT[_level] = _report

    print(f"=== {_level.upper()} ===")
    print(f"  {len(_gdf)} zones, CRS {_gdf.crs.to_string()} "
          f"(projected: {not _gdf.crs.is_geographic})")
    print(f"  scheme {_report['scheme']}, transform {_report['transform']}")
    print(f"  neighbours: min {_report['min_neighbours']}, "
          f"mean {_report['mean_neighbours']:.2f}, max {_report['max_neighbours']}")
    print(f"  ISLANDS before repair: {_report['islands_before_repair']}"
          f" -> repaired {_report['islands_repaired']} -> remaining {_report['islands']}")
    if _report["islands_before_repair"]:
        print(f"  island zones were: {_report['island_ids']}")
    if _report["islands"]:
        print("  *** ISLANDS REMAIN. Their local statistics are computed over an")
        print("  *** empty neighbourhood and are emitted as NaN, not as zero.")
    print()

# Committed alongside the tables: the weights are part of the method, and a
# cluster map cannot be reproduced or audited without the adjacency behind it.
for _level, _gdf in ZONES.items():
    _out = f"data/outputs/{_level}_divisions_colombo.geojson"
    _gdf.to_file(_out, driver="GeoJSON")
    print("Wrote", _out, f"({os.path.getsize(_out) / 1024:.0f} KB)")
    _cap = float(SS["geometry"]["max_geojson_mb"]) * 1024
    if os.path.getsize(_out) / 1024 > _cap:
        print(f"  *** OVER the {SS['geometry']['max_geojson_mb']} MB budget - raise")
        print("  *** spatial_stats.geometry.simplify_m and re-export rather than")
        print("  *** committing it.")

## Step 7 - global Moran's I: is there any structure at all?

The question every local statistic presupposes. If urban LST at 1 km2 units were
not autocorrelated, there would be no clusters to find and the rest of the
notebook would be pattern-hunting in noise.

Reported with **both** inferential routes: the normality approximation and a
999-draw permutation. When they disagree, trust the permutation.

In [ ]:
# COLAB: RUN THIS CELL
COVARIATES = {}
for (_level, _epoch), _path in COVARIATE_CSV.items():
    if _path is None:
        continue
    COVARIATES[(_level, _epoch)] = spatial_stats.read_zone_covariates(
        _path, params, _level
    )

_rows = []
for (_level, _epoch), _frame in sorted(COVARIATES.items()):
    _aligned = _frame.set_index("zone_id").reindex(ZONE_IDS[_level])
    for _column in [RESPONSE, *PREDICTORS]:
        _values = _aligned[_column].to_numpy(dtype="float64")
        if not np.isfinite(_values).all():
            _rows.append({"level": _level, "epoch": _epoch, "variable": _column,
                          "status": "incomplete",
                          "n_missing": int((~np.isfinite(_values)).sum())})
            continue
        _result = spatial_stats.global_morans_i(_values, WEIGHTS[_level], params)
        _rows.append({"level": _level, "epoch": _epoch, "variable": _column,
                      "status": "ok", "n": _result["n"], "morans_i": _result["i"],
                      "expectation": _result["expectation"],
                      "z_norm": _result["z_norm"], "p_norm": _result["p_norm"],
                      "z_sim": _result["z_sim"], "p_sim": _result["p_sim"]})

MORANS = pd.DataFrame(_rows)
MORANS.to_csv("data/outputs/morans_i_global_2000_2025.csv", index=False)
print("Wrote data/outputs/morans_i_global_2000_2025.csv")
display(MORANS[MORANS["variable"] == RESPONSE])

_headline = MORANS[(MORANS["variable"] == RESPONSE) & (MORANS["level"] == "gn")
                   & (MORANS["status"] == "ok")]
if not _headline.empty:
    _worst = _headline["p_sim"].max()
    print()
    if (_headline["morans_i"] > 0).all() and _worst <= 0.01:
        print(f"PASS: LST is positively autocorrelated at GN level in every epoch "
              f"(worst p_sim = {_worst:.3g}).")
    else:
        print("*** CHECK THE JOIN. Urban LST at ~1.25 km2 units should be strongly")
        print("*** positively autocorrelated. A weak or negative Moran's I here")
        print("*** usually means the covariate table and the weights are in")
        print("*** different zone orders, not that Colombo is unusual.")

## Step 8 - LISA cluster maps

Local Moran's I splits the map into four kinds of place, and the distinction
between two of them is the one most often lost:

* **HH / LL** are *clusters* - a hot division among hot neighbours, or cool
  among cool.
* **HL / LH** are *spatial outliers* - a hot division surrounded by cool ones.
  These are not small hot spots. They are discontinuities, and they are often
  the most interesting units on the map.

Significance is the **Benjamini-Hochberg adjusted** permutation p. The raw count
is printed beside it: with 557 simultaneous tests at alpha 0.05, about 28
"clusters" are expected from noise alone, and the gap between the two counts is
the honest measure of that.

In [ ]:
# COLAB: RUN THIS CELL
LISA = {}
for _epoch in EPOCHS:
    _key = ("gn", _epoch)
    if _key not in COVARIATES:
        print(f"{_epoch}: covariates missing, skipping")
        continue
    _aligned = COVARIATES[_key].set_index("zone_id").reindex(ZONE_IDS["gn"])
    _values = _aligned[RESPONSE].to_numpy(dtype="float64")
    if not np.isfinite(_values).all():
        print(f"{_epoch}: {int((~np.isfinite(_values)).sum())} zone(s) have no LST; "
              "LISA needs a complete surface, skipping")
        continue

    _frame = spatial_stats.local_morans(
        _values, WEIGHTS["gn"], params, zone_ids=ZONE_IDS["gn"]
    )
    LISA[_epoch] = _frame
    _out = f"data/outputs/lisa_gn_{_epoch}.csv"
    _frame.to_csv(_out, index=False)

    _raw = int((_frame["p_sim"] < SS["lisa"]["alpha"]).sum())
    _adj = int(_frame["significant"].sum())
    print(f"{_epoch}: {_adj} significant after FDR (raw {_raw} at alpha "
          f"{SS['lisa']['alpha']}) -> {_out}")
    print("   ", _frame[_frame["significant"]]["cluster"].value_counts().to_dict())

    _fig = viz.plot_cluster_map(
        ZONES["gn"], _frame, f"figures/lisa_gn_{_epoch}.png", params,
        title=f"LISA clusters of dry-season LST, {_epoch} ({EPOCH_SOURCE}, GN)",
    )
    print("    wrote", _fig)
    display(Image(filename=str(_fig)))

## Step 9 - Getis-Ord Gi\* hot and cold spots

Gi\* answers a different question from LISA: not "is this unit unlike its
neighbours" but "is this *neighbourhood* unusually hot for the city as a whole".
It uses **binary weights with the focal unit included** - row-standardising it
would force every neighbourhood sum to 1 and collapse the variance term that
lets a large, well-connected neighbourhood outweigh a small one.

It also runs at **both** aggregation levels, which is the first row of the MAUP
comparison assembled in Step 12.

The last cell re-runs the final epoch on the single-sensor series. **If the
cluster geography changes, decision 1 in the header is wrong and the epoch maps
must be re-scoped.**

In [ ]:
# COLAB: RUN THIS CELL
HOTSPOTS = {}
for _level in LEVELS:
    if _level not in WEIGHTS:
        continue
    for _epoch in EPOCHS:
        _key = (_level, _epoch)
        if _key not in COVARIATES:
            continue
        _aligned = COVARIATES[_key].set_index("zone_id").reindex(ZONE_IDS[_level])
        _values = _aligned[RESPONSE].to_numpy(dtype="float64")
        if not np.isfinite(_values).all():
            print(f"{_level}/{_epoch}: incomplete LST surface, skipping")
            continue

        _frame = spatial_stats.gi_star(
            _values, WEIGHT_REPORT[_level]["binary_matrix"], params,
            zone_ids=ZONE_IDS[_level],
        )
        HOTSPOTS[_key] = _frame
        _out = f"data/outputs/gi_star_{_level}_{_epoch}.csv"
        _frame.to_csv(_out, index=False)
        _hot = int(_frame["confidence_class"].str.startswith("hot").sum())
        _cold = int(_frame["confidence_class"].str.startswith("cold").sum())
        print(f"{_level}/{_epoch}: {_hot} hot, {_cold} cold of {len(_frame)} -> {_out}")

        _fig = viz.plot_hotspot_map(
            ZONES[_level], _frame, f"figures/gi_star_{_level}_{_epoch}.png", params,
            title=f"Getis-Ord Gi* of dry-season LST, {_epoch} ({_level.upper()})",
        )
        if _level == "gn":
            display(Image(filename=str(_fig)))

In [ ]:
# COLAB: RUN THIS CELL
# THE SENSITIVITY CHECK for decision 1. Same epoch, same zones, single-sensor
# series. The cluster geography must be essentially unchanged.
_name = exports.export_name(
    "zone_covariates", "district", params, res_m=EPOCH_SCALE,
    suffix=SENSITIVITY_SUFFIX,
)
_sensitivity_path = f"data/interim/{_name}.csv"
_pooled_key = ("gn", EPOCHS[-1])

if not os.path.exists(_sensitivity_path):
    print(f"{_sensitivity_path} not downloaded. The pooled-series decision is")
    print("then ARGUED but not TESTED - say so rather than leaving it implied.")
elif _pooled_key not in HOTSPOTS:
    print("the pooled 2020s Gi* run is unavailable, so there is nothing to")
    print("compare the sensitivity run against.")
else:
    _alt_frame = spatial_stats.read_zone_covariates(_sensitivity_path, params, "gn")
    _aligned = _alt_frame.set_index("zone_id").reindex(ZONE_IDS["gn"])
    _values = _aligned[RESPONSE].to_numpy(dtype="float64")
    if not np.isfinite(_values).all():
        print(f"{int((~np.isfinite(_values)).sum())} zone(s) have no LST on the")
        print(f"{SENSITIVITY_SOURCE} series (it starts in 2014), so Gi* cannot be")
        print("computed on a complete surface. Report the coverage gap instead.")
    else:
        _alt_gi = spatial_stats.gi_star(
            _values, WEIGHT_REPORT["gn"]["binary_matrix"], params,
            zone_ids=ZONE_IDS["gn"],
        )
        _pooled = HOTSPOTS[_pooled_key]
        _agree = float((_alt_gi["confidence_class"].to_numpy()
                        == _pooled["confidence_class"].to_numpy()).mean())
        _corr = float(np.corrcoef(_alt_gi["gi_z"], _pooled["gi_z"])[0, 1])
        pd.DataFrame({
            "zone_id": ZONE_IDS["gn"],
            "gi_z_pooled": _pooled["gi_z"].to_numpy(),
            "gi_z_single_sensor": _alt_gi["gi_z"].to_numpy(),
            "class_pooled": _pooled["confidence_class"].to_numpy(),
            "class_single_sensor": _alt_gi["confidence_class"].to_numpy(),
        }).to_csv("data/outputs/gi_star_sensor_sensitivity_gn.csv", index=False)
        print(f"class agreement {_agree:.1%} | Gi* z correlation {_corr:.3f}")
        print("Wrote data/outputs/gi_star_sensor_sensitivity_gn.csv")
        print()
        if _agree > 0.85 and _corr > 0.9:
            print("PASS: the cluster geography does not depend on which Landsat")
            print("series is pooled, so the within-epoch argument holds here")
            print("EMPIRICALLY and not merely in principle.")
        else:
            print("*** The cluster geography DOES change with the series. Do not")
            print("*** use the pooled epoch maps; re-scope Steps 8 and 9 onto")
            print(f"*** {SENSITIVITY_SOURCE} and accept the shorter record.")

## Step 10 - Emerging Hot Spot Analysis

Implemented here in Python, not assumed from ArcGIS. Three stages:

1. **Space-time bins** - one bin per year. Every zone must appear in every bin,
   because the panel is later multiplied by a positional weights matrix and a
   hole would give every subsequent zone its neighbour's statistic.
2. **Gi\* per bin** - each bin is standardised against its own spatial mean, so
   a sensor step or a hot year raises every zone together and cancels. What
   survives is the change in *pattern*.
3. **Mann-Kendall across bins**, then the category rules.

**And then the part that matters.** Over 12 bins a Mann-Kendall test cannot
resolve most real trends, so "no pattern" may mean *stable* or *too short to
tell*. Every zone therefore ships its own detection limit and an `underpowered`
flag, and the map hatches those zones. This is the same distinction that turned
Phase 4's Landsat zero from an absence into a bound.

In [ ]:
# COLAB: RUN THIS CELL
EHSA = {}
for (_source, _level), _series in sorted(EHSA_SERIES.items()):
    if _level not in WEIGHTS:
        continue
    _panel = spatial_stats.space_time_bins(_series, params)
    # Report the alignment BEFORE reindexing. If the series' zone ids and the
    # geometry's zone ids do not overlap - different key, different dtype - the
    # reindex produces an all-NaN panel, every bin is then dropped, and the only
    # symptom is an empty result several lines later.
    _overlap = len(set(map(str, _panel.index)) & set(map(str, ZONE_IDS[_level])))
    print(f"{_source}/{_level}: {_overlap} of {len(ZONE_IDS[_level])} zones matched "
          f"between the series and the geometry")
    if _overlap < 0.9 * len(ZONE_IDS[_level]):
        print("  *** POOR OVERLAP. Both sides must be keyed on the pcode; check")
        print("  *** that the series' zone_id is adm4_pcode and not a name.")
    _panel = _panel.reindex(ZONE_IDS[_level])
    _names = dict(zip(_series["zone_id"], _series["name"]))

    with warnings.catch_warnings():
        warnings.simplefilter("always")
        _gi_panel = spatial_stats.gi_star_panel(
            _panel, WEIGHT_REPORT[_level]["binary_matrix"], params,
            zone_ids=ZONE_IDS[_level],
        )
    if _gi_panel.empty:
        print(f"{_source}/{_level}: no complete time bin, skipping")
        continue

    _frame = spatial_stats.classify_emerging_hotspots(_gi_panel, params, names=_names)
    EHSA[(_source, _level)] = _frame
    _out = f"data/outputs/ehsa_{_level}_{_source}.csv"
    _frame.to_csv(_out, index=False)

    _verdict = spatial_stats.ehsa_power_check(_frame, params)
    print(f"=== {_source} / {_level.upper()} ===")
    print(f"  {_gi_panel.shape[1]} usable bins of {_panel.shape[1]}"
          + (f" (dropped {_gi_panel.attrs['dropped_bins']})"
             if _gi_panel.attrs.get("dropped_bins") else ""))
    print("  categories:", _frame["category"].value_counts().to_dict())
    print(" ", _verdict["verdict"])
    print("  ->", _out)

    _fig = viz.plot_ehsa_map(
        ZONES[_level], _frame, f"figures/ehsa_{_level}_{_source}.png", params,
        title=f"Emerging hot spots, {_source} ({_level.upper()}, "
              f"{_gi_panel.shape[1]} annual bins)",
    )
    if _level == "gn":
        display(Image(filename=str(_fig)))
    print()

print("REPORT THE TWO SERIES AS A PAIR. landsat_oli_dry has the intra-urban")
print("detail and 12 bins; terra_night has 26 bins and one sensor but 1 km")
print("pixels against ~1.25 km2 divisions. Neither is sufficient alone.")

## Step 11 - the driver-attribution ladder

CLAUDE.md's escalation path, run in order and with each rung's result deciding
the next:

`OLS` -> `Moran's I on the residuals` -> `Lagrange Multiplier tests` ->
`spatial lag or error` -> `GWR` -> `MGWR`

The LM decision rule is encoded, not eyeballed, so the model choice is
reproducible and can be stated in the report as a rule rather than a judgement.

Expect the VIF table to fire on NDVI against NDBI. Phase 3 already measured the
symptom: NDVI's partial coefficient flipped sign in 5 of 26 years while its
bivariate correlation stayed a clean -0.51. That collinearity is precisely why
the ladder ends at MGWR rather than at a single multivariate coefficient.

In [ ]:
# COLAB: RUN THIS CELL
MODEL_EPOCH = EPOCHS[-1]
LADDER = {}

for _level in LEVELS:
    _key = (_level, MODEL_EPOCH)
    if _key not in COVARIATES or _level not in WEIGHTS:
        continue
    print(f"================ {_level.upper()} / {MODEL_EPOCH} ================")

    _frame = spatial_stats.build_model_frame([COVARIATES[_key]], params)
    _frame = _frame.set_index("zone_id").reindex(ZONE_IDS[_level]).dropna()
    _keep = [ZONE_IDS[_level].index(z) for z in _frame.index]
    _matrix = WEIGHTS[_level][np.ix_(_keep, _keep)]
    _frame = _frame.reset_index()
    print(f"{len(_frame)} complete zones of {len(ZONE_IDS[_level])}")

    _gate = spatial_stats.require_estimable(
        len(_frame), len(PREDICTORS), params, statistic="the local models"
    )
    print(f"estimable: {_gate['estimable']} (needs {_gate['required']})")
    if not _gate["estimable"]:
        print("  " + _gate["reason"])

    _y = _frame[RESPONSE].to_numpy(dtype="float64")
    _X = _frame[PREDICTORS].to_numpy(dtype="float64")

    _vif = spatial_stats.variance_inflation_factors(_X, PREDICTORS)
    print("\nVIF:")
    display(_vif)
    if (_vif["vif"] > SS["regression"]["vif_warn"]).any():
        print(f"  Collinear predictors above VIF {SS['regression']['vif_warn']}. "
              "Partial coefficients are unstable;")
        print("  quote the GWR/MGWR surfaces and the bivariate correlations, not a "
              "single global beta.")

    _ols = spatial_stats.ols_fit(_y, _X, PREDICTORS)
    print(f"\nOLS  R2={_ols['r_squared']:.3f}  adj={_ols['adj_r_squared']:.3f}")
    display(_ols["coefficients"])

    _diag = spatial_stats.lagrange_multiplier_tests(_ols, _y, _matrix)
    print(f"residual Moran's I = {_diag['moran_i']:.4f}  z={_diag['moran_z']:.3f}  "
          f"p={_diag['moran_p']:.3g}")
    for _test in ("lm_lag", "lm_error", "rlm_lag", "rlm_error", "lm_sarma"):
        print(f"  {_test:10s} stat={_diag[_test][0]:9.3f}  p={_diag[_test][1]:.4g}")

    _decision = spatial_stats.lm_decision(_diag, params)
    print(f"\nDECISION (rule {_decision['rule']}): fit the {_decision['model'].upper()} "
          f"model - {_decision['reason']}")

    # NOT _try_ee: that only catches ee.EEException, and a maximum-likelihood
    # fit fails with a linear-algebra or convergence error instead, which would
    # abort the whole run rather than degrade.
    _spatial = None
    if _decision["model"] in ("lag", "error"):
        _fit = (spatial_stats.spatial_lag_model if _decision["model"] == "lag"
                else spatial_stats.spatial_error_model)
        try:
            _spatial = _fit(_frame, _matrix, params)
        except Exception as _error:
            print(f"\nthe {_decision['model']} model failed to fit: {_error}")
            print(f"Try spatial_stats.regression.lag_estimator: 'gm' - the "
                  "generalised-method-of-moments estimator does not need the")
            print("log-determinant that maximum likelihood struggles with.")
    if _spatial is not None:
        print(f"\n{_spatial['model']} model ({_spatial['estimator']}), "
              f"pseudo R2 = {_spatial['pseudo_r_squared']:.3f}")
        display(_spatial["coefficients"])
        _spatial["coefficients"].to_csv(
            f"data/outputs/spatial_{_spatial['model']}_{_level}_{MODEL_EPOCH}.csv",
            index=False,
        )

    _ols["coefficients"].to_csv(
        f"data/outputs/ols_{_level}_{MODEL_EPOCH}.csv", index=False)
    _vif.to_csv(f"data/outputs/vif_{_level}_{MODEL_EPOCH}.csv", index=False)
    LADDER[_level] = {
        "frame": _frame, "matrix": _matrix, "ols": _ols, "diagnostics": _diag,
        "decision": _decision, "vif": _vif, "gate": _gate, "spatial": _spatial,
    }
    print()

In [ ]:
# COLAB: RUN THIS CELL
# GWR and MGWR. The estimability gate decides whether this runs at all, and a
# refusal is recorded rather than worked around.
GWR_RESULTS = {}

for _level, _state in LADDER.items():
    if not _state["gate"]["estimable"]:
        print(f"{_level.upper()}: SKIPPED - {_state['gate']['reason']}")
        print("  This is the aggregation-unit finding, not a gap in the run.\n")
        continue

    _frame = _state["frame"]
    _gdf = ZONES[_level].set_index("zone_id").reindex(_frame["zone_id"])
    _coords = np.column_stack([_gdf.geometry.centroid.x, _gdf.geometry.centroid.y])

    for _multiscale, _label in ((False, "gwr"), (True, "mgwr")):
        _t0 = time.time()
        try:
            _result = spatial_stats.gwr_model(
                _frame, _coords, params, multiscale=_multiscale
            )
        except Exception as _error:
            print(f"{_level}/{_label} failed: {_error}")
            continue
        GWR_RESULTS[(_level, _label)] = _result
        print(f"{_level}/{_label}: R2={_result['r_squared']:.3f} "
              f"AICc={_result['aicc']:.1f} critical t={_result['critical_t']:.3f} "
              f"in {time.time() - _t0:.0f} s")
        if _multiscale:
            print("  bandwidths:", dict(zip(_result["terms"], _result["bandwidth"])))
            if len(set(_result["bandwidth"])) == 1:
                print("  *** every bandwidth is identical: the multiscale search")
                print("  *** collapsed and this result is just GWR. Say so.")
            else:
                print("  A SMALL bandwidth means that relationship is LOCAL; one")
                print("  near n means it is effectively global.")
        else:
            print(f"  bandwidth: {_result['bandwidth']}")

        _out = f"data/outputs/{_label}_local_coefficients_{_level}_{MODEL_EPOCH}.csv"
        _result["local_coefficients"].to_csv(_out, index=False)
        _fig = viz.plot_gwr_coefficients(
            ZONES[_level], _result["local_coefficients"],
            f"figures/{_label}_local_coefficients_{_level}_{MODEL_EPOCH}.png", params,
            title=f"{_label.upper()} local coefficients, {MODEL_EPOCH} ({_level.upper()})",
        )
        display(Image(filename=str(_fig)))

    if (_level, "mgwr") in GWR_RESULTS:
        _bw = GWR_RESULTS[(_level, "mgwr")]
        pd.DataFrame({"term": _bw["terms"], "bandwidth": _bw["bandwidth"]}).to_csv(
            f"data/outputs/mgwr_bandwidths_{_level}_{MODEL_EPOCH}.csv", index=False)

## Step 12 - the MAUP comparison

CLAUDE.md requires aggregation-unit sensitivity to be reported rather than a
single number. Phase 3 already measured it on the means: the same surface has
sd 2.74 and range 12.65 at GN against sd 2.62 and range **7.98** at DS - coarser
units average the extremes away, so a DS hot-spot map is not a downsampled GN
one.

This table extends that to the statistics themselves, and **a statistic that
could not be estimated at a level is a row with a reason, never an omission**.
Which methods survive the coarsening from 557 units to 13 is itself the finding.

In [ ]:
# COLAB: RUN THIS CELL
_records = []

for _level in LEVELS:
    _n = len(ZONE_IDS.get(_level, []))
    if not _n:
        continue

    _row = MORANS[(MORANS["level"] == _level) & (MORANS["epoch"] == MODEL_EPOCH)
                  & (MORANS["variable"] == RESPONSE)]
    if not _row.empty and _row.iloc[0]["status"] == "ok":
        _records.append({
            "statistic": "global_morans_i", "level": _level, "n_units": _n,
            "value": float(_row.iloc[0]["morans_i"]),
            "detail": f"p_sim={_row.iloc[0]['p_sim']:.4g}, z_norm={_row.iloc[0]['z_norm']:.2f}",
        })

    _key = (_level, MODEL_EPOCH)
    if _key in HOTSPOTS:
        _frame = HOTSPOTS[_key]
        _records.append({
            "statistic": "gi_star_significant_zones", "level": _level, "n_units": _n,
            "value": float((_frame["confidence_class"] != "ns").sum()),
            "detail": f"{float((_frame['confidence_class'] != 'ns').mean()):.1%} of zones, FDR adjusted",
        })

    if _level in LISA or _level == "gn":
        _frame = LISA.get(MODEL_EPOCH)
        if _frame is not None and _level == "gn":
            _records.append({
                "statistic": "lisa_significant_zones", "level": _level, "n_units": _n,
                "value": float(_frame["significant"].sum()),
                "detail": f"raw {int((_frame['p_sim'] < SS['lisa']['alpha']).sum())} before FDR",
            })
    if _level == "ds":
        _records.append({
            "statistic": "lisa_significant_zones", "level": _level, "n_units": _n,
            "status": spatial_stats.MAUP_NOT_ESTIMABLE,
            "reason": ("a local statistic over 13 units rests on 2-5 neighbours per "
                       "zone; the conditional-randomisation reference set is smaller "
                       "than the permutation count, so the pseudo p-value is not a "
                       "test but a lookup"),
        })

    _state = LADDER.get(_level)
    if _state is not None:
        _records.append({
            "statistic": "ols_r_squared", "level": _level, "n_units": len(_state["frame"]),
            "value": float(_state["ols"]["r_squared"]),
            "detail": f"residual Moran's I {_state['diagnostics']['moran_i']:.3f} "
                      f"(p={_state['diagnostics']['moran_p']:.3g}); "
                      f"LM points to {_state['decision']['model'].upper()}",
        })
        for _label in ("gwr", "mgwr"):
            _result = GWR_RESULTS.get((_level, _label))
            if _result is not None:
                _detail = (f"bandwidth {_result['bandwidth']}"
                           if _label == "gwr"
                           else f"bandwidths {dict(zip(_result['terms'], _result['bandwidth']))}")
                _records.append({
                    "statistic": _label, "level": _level, "n_units": _result["n"],
                    "value": float(_result["r_squared"]), "detail": _detail,
                })
            elif not _state["gate"]["estimable"]:
                _records.append({
                    "statistic": _label, "level": _level, "n_units": len(_state["frame"]),
                    "status": spatial_stats.MAUP_NOT_ESTIMABLE,
                    "reason": _state["gate"]["reason"],
                })

MAUP = spatial_stats.maup_comparison(_records, params)
MAUP.to_csv("data/outputs/maup_comparison_2000_2025.csv", index=False)
print("Wrote data/outputs/maup_comparison_2000_2025.csv")
display(MAUP)

_fig = viz.plot_maup_table(MAUP, "figures/maup_comparison_2000_2025.png", params)
print("Wrote", _fig)
display(Image(filename=str(_fig)))

_refused = MAUP[MAUP["status"] != "ok"]
print()
print(f"{len(_refused)} of {len(MAUP)} statistic/level combinations are NOT")
print("estimable. Quote that as the aggregation-unit result: the DS map is not a")
print("coarser version of the GN map, it is a different and smaller set of")
print("answerable questions.")

## Step 13 - landscape metrics on the green-space class

Patch density, edge density, mean patch size and the aggregation index,
implemented in `scipy.ndimage` rather than delegated to R, so they are covered
by the same pytest suite as the rest of the pure-Python core.

**Every one of these is scale dependent.** The same city at 10 m and at 30 m
gives different patch counts, a different edge density and a different
aggregation index. The grid size is returned with the metrics and must be quoted
with any figure.

Zones are rasterised locally from the committed GeoJSON onto the exported
raster's own grid, so the per-zone metrics are computed on exactly the pixels the
class raster has.

In [ ]:
# COLAB: RUN THIS CELL
from rasterio import features as rio_features

_records = []
_zone_records = []
_cell = float(SS["landscape"]["raster_scale_m"])

for (_scheme, _year), _path in sorted(GREEN_TIF.items()):
    if _path is None:
        print(f"{_scheme} {_year}: raster not downloaded, skipping")
        continue
    with rasterio.open(_path) as _src:
        _band = _src.read(1)
        _nodata = _src.nodata
        _transform = _src.transform
        _crs = _src.crs
        _shape = _band.shape
        _cell_m = abs(_transform.a)

    _valid = np.ones(_shape, dtype=bool) if _nodata is None else (_band != _nodata)
    _green = (_band == 1) & _valid
    _metrics = spatial_stats.landscape_metrics(_green, _cell_m, params, valid=_valid)
    _metrics.update({"scheme": _scheme, "year": _year, "zone_id": None})
    _records.append(_metrics)
    print(f"{_scheme} {_year} at {_cell_m:g} m: "
          f"{_metrics['class_area_ha']:,.0f} ha green "
          f"({_metrics['class_fraction']:.1%}), {int(_metrics['n_patches']):,} patches, "
          f"AI {_metrics['aggregation_index_pct']:.1f}%")

    if SS["landscape"]["per_zone"] and "gn" in ZONES:
        _gdf = ZONES["gn"].to_crs(_crs)
        _codes = {i + 1: z for i, z in enumerate(_gdf["zone_id"])}
        _zone_raster = rio_features.rasterize(
            ((geom, code) for code, geom in zip(_codes, _gdf.geometry)),
            out_shape=_shape, transform=_transform, fill=0, dtype="int32",
        )
        _zone_frame = spatial_stats.landscape_metrics_by_zone(
            np.where(_green, 1, 0), _zone_raster, params, _cell_m, [1],
            zone_labels=_codes, scheme=_scheme, year=_year,
        )
        _zone_records.append(_zone_frame)
        print(f"    per-zone metrics for {len(_zone_frame)} GN divisions")

LANDSCAPE = spatial_stats.build_landscape_frame(_records, params)
LANDSCAPE.to_csv("data/outputs/landscape_metrics_green.csv", index=False)
print("\nWrote data/outputs/landscape_metrics_green.csv")
display(LANDSCAPE)

if _zone_records:
    LANDSCAPE_BY_ZONE = pd.concat(_zone_records, ignore_index=True)
    LANDSCAPE_BY_ZONE.to_csv("data/outputs/landscape_metrics_green_by_gn.csv", index=False)
    print("Wrote data/outputs/landscape_metrics_green_by_gn.csv "
          f"({len(LANDSCAPE_BY_ZONE)} rows) - Phase 7 consumes this as a "
          "fragmentation criterion")

if not LANDSCAPE.empty:
    _fig = viz.plot_landscape_change(
        LANDSCAPE, "figures/landscape_metrics_green.png", params)
    print("Wrote", _fig)
    display(Image(filename=str(_fig)))

    _dw = LANDSCAPE[LANDSCAPE["scheme"] == "dynamic_world"].sort_values("year")
    if len(_dw) >= 2:
        _first, _last = _dw.iloc[0], _dw.iloc[-1]
        print(f"\nDynamic World {int(_first['year'])} -> {int(_last['year'])}:")
        print(f"  green area   {_first['class_area_ha']:,.0f} -> "
              f"{_last['class_area_ha']:,.0f} ha")
        print(f"  patch density {_first['patch_density_per_100ha']:.2f} -> "
              f"{_last['patch_density_per_100ha']:.2f} per 100 ha")
        print(f"  aggregation   {_first['aggregation_index_pct']:.1f} -> "
              f"{_last['aggregation_index_pct']:.1f} %")
        print("  Rising patch density with falling aggregation is FRAGMENTATION:")
        print("  the same or less green space broken into more, smaller pieces.")
        print("  Compare against the WorldCover row: a large gap there means the")
        print("  metric is partly measuring the classifier, not the city.")

## Step 14 - BRING THE RESULTS HOME

**Do not skip this.** Everything Part 2 produced lives in a Colab VM that will be
recycled, and `CLAUDE.md` requires `data/outputs/` to be committed. That includes
the two GeoJSONs, which are the only record of the geometry every statistic here
was computed on.

The GeoTIFFs and the exported CSVs in `data/interim/` are deliberately excluded:
they are large and fully reproducible from the Drive exports. The derived tables
and figures are not reproducible without another full run.

In [ ]:
# COLAB: RUN THIS CELL
# Bundle every committable artefact for download, then commit it locally.
import zipfile

_bundle = "/content/phase5_outputs.zip"
_written = []
with zipfile.ZipFile(_bundle, "w", zipfile.ZIP_DEFLATED) as _zip:
    for _folder in ("data/outputs", "figures"):
        for _root, _, _files in os.walk(_folder):
            for _name in _files:
                if _name == ".gitkeep":
                    continue
                _path = os.path.join(_root, _name)
                _zip.write(_path, _path)
                _written.append(_path)

print(f"{len(_written)} file(s) bundled into {_bundle}:")
for _path in sorted(_written):
    print("  ", _path, f"({os.path.getsize(_path) / 1024:.0f} KB)")
print()
if not _written:
    print("NOTHING TO BUNDLE - run Parts 1 and 2 first.")
else:
    print("Download it from the Colab Files pane (folder icon, left sidebar),")
    print("unzip into the repo root locally, then:")
    print("    git add data/outputs figures && git commit && git push")
    try:
        from google.colab import files
        files.download(_bundle)
    except Exception as _error:
        print()
        print(f"(auto-download unavailable: {_error} - use the Files pane)")

## What to check before signing Phase 5 off

Report the answers back so they can be recorded in `PROGRESS.md`, the way
Phases 1-4 were closed.

### Must pass

1. **Step 1 probe** - the largest `abs_diff` against `esda` and `spreg` is
   essentially zero, and the LISA quadrant coding matches. Paste both tables.
   This is the evidence that the analytic implementations are sound; without it
   every map below is unverified.
2. **Zone counts** - 557 GN and 13 DS, no duplicate `zone_id`, both GeoJSONs
   read back in EPSG:32644 and under the size budget.
3. **Island count** - how many GN divisions had no queen neighbour, and how many
   remained after repair. Zero remaining is required. If any remain, name them.
4. **Global Moran's I on 2020s GN LST is positive and significant.** If it is
   not, the covariate table and the weights are almost certainly in different
   zone orders - that is far more likely than Colombo being unusual.
5. **`pytest tests/test_spatial_stats.py -q` inside Colab**, where PySAL is
   actually installed. Three tests skip locally for want of `esda`, `libpysal`
   and `spreg`; they are the cross-validation tests and they must pass here.
   Also re-run `pytest tests/ -q` and confirm 736 passing.

### Judgement calls to report, not to fix silently

6. **The Step 9 sensitivity check.** Does the 2020s cluster geography survive
   swapping the pooled Landsat series for the single-sensor one? If not, the
   epoch maps must be re-scoped and the header decision is wrong.
7. **The LM decision** at GN: which model did the rule select, and was the
   residual Moran's I significant? A non-significant residual Moran's I would
   mean the spatial models were unnecessary - itself a reportable result.
8. **MGWR bandwidths.** Are they different per covariate? Identical bandwidths
   mean the multiscale search collapsed and the result is only GWR.
9. **EHSA power.** What fraction of "no pattern" zones are flagged
   `underpowered`, for each of the two series? Over the 12-bin Landsat panel a
   high fraction is expected and is the honest headline, not a disappointment.
10. **Landscape metrics.** Do Dynamic World 2016 and 2024 differ in a plausible
    direction, and how far does WorldCover 2021 sit from the Dynamic World
    trajectory? A large gap means the metric is partly measuring the classifier.

### Known limits that travel into Phase 6

* These statistics describe **polygons**. A zonal coefficient is not a
  pixel-level or person-level relationship.
* The epoch cluster maps carry **no** epoch-to-epoch magnitude. Only Phase 4's
  Sen's slope measures change.
* Population is the **2020** WorldPop layer and built fraction is a **5-year
  GHSL epoch**, whatever epoch they sit beside.
* Gi\* and LISA are computed on dry-season LST at a single ~10:30 overpass;
  the night-time pattern comes only from the MODIS EHSA run.